# Initialization

In [3]:
import json

import pandas as pd
import ast
import os
from tqdm import tqdm

from typing import Union, List
tqdm.pandas()

# Load Impact DF

In [4]:
def try_literal_eval(x):
    if isinstance(x, str):
        x = x.strip()
        if (x.startswith("[") and x.endswith("]")) or \
           (x.startswith("{") and x.endswith("}")) or \
           (x.startswith("(") and x.endswith(")")):
            try:
                return ast.literal_eval(x)
            except (ValueError, SyntaxError):
                return x
    return x

In [5]:
dataset_dir = "/home/yishin/keith/patent_research/model_io"

In [6]:
Impact_df = pd.read_csv(os.path.join(dataset_dir, "Impact_Sub.csv"), encoding="utf-8")
Impact_df = Impact_df.map(try_literal_eval)

Impact_df.head()

,title,caption,image_paths,best_fig_desc,Loc_class,main_class,sub_class
0,Cupcake with contrasting icing,The image is a black and white picture of a cu...,[impact_dataset/2022/USD0957089-20220712/USD09...,{'FIG. 1': 'FIG. 1 is a perspective view of a ...,"{11-01, 01-01}",1,1
1,Ring with an integrated spoon,The image is a drawing of a ring with an integ...,[impact_dataset/2022/USD0966130-20221011/USD09...,{'FIG. 1': 'FIG. 1 is a front perspective view...,"{11-01, 01-01}",1,1
2,Butter stick,The image is a white and black drawing of a bu...,[impact_dataset/2022/USD0962586-20220906/USD09...,"{'FIG. 1': 'FIG. 1 is a front, top and left si...","{11-01, 01-06}",1,6
3,Necklace,"The image is a round shape, and the necklace i...",[impact_dataset/2022/USD0962109-20220830/USD09...,{'FIG. 3': 'FIG. 3 is a right side perspective...,"{11-01, 01-01}",1,1
4,Dietary supplement,"The image is circular in shape, and it represe...",[impact_dataset/2022/USD0941457-20220118/USD09...,{'FIG. 1': 'FIG. 1 is a top perspective view o...,"{11-01, 01-01}",1,1


# Select Best Figures

In [67]:
import pytesseract
import re
import numpy as np
import cv2
from pathlib import Path
from PIL import Image, ImageDraw, ImageFilter, ImageFont, ImageOps

In [ ]:
def preprocess_for_ocr(image: Image.Image, min_height: int, dilate: bool = True) -> Image.Image:
    image = image.convert("L")
    image = ImageOps.autocontrast(image)

    w, h = image.size
    scale = max(1.0, min_height / h)
    if scale > 1.0:
        image = image.resize((int(w * scale), int(h * scale)), Image.Resampling.LANCZOS)

    arr = np.array(image)

    # Denoise before binarization — removes scanner noise without blurring text
    arr = cv2.fastNlMeansDenoising(arr, h=15)

    # Otsu picks the threshold automatically from the image histogram
    _, arr = cv2.threshold(arr, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    if dilate:
        kernel = np.ones((2, 2), np.uint8)
        arr = cv2.dilate(arr, kernel, iterations=1)

    # Padding helps Tesseract with text near the image edge
    arr = cv2.copyMakeBorder(arr, 20, 20, 20, 20, cv2.BORDER_CONSTANT, value=255)

    return Image.fromarray(arr)

In [ ]:
def _normalise_label(raw_num: str) -> str:
    """'1 a' → 'FIG. 1A', '12B' → 'FIG. 12B'"""
    num = re.sub(r'\s+', '', raw_num).upper()
    return f"FIG. {num}"

In [84]:
_FIG_LABEL = re.compile(
    r'\bF[I1lL|]G[S]?[.\s_\-·,]?\s*(\d{1,3}\s*[A-Za-z]?)\b',
    re.IGNORECASE,
)
_CONFIGS = [r'--oem 3 --psm 11', r'--oem 3 --psm 6', r'--oem 3 --psm 3']

def extract_fig_labels(img_path: str, min_height: int = 800) -> list[tuple[str, int]]:
    pil_img = Image.open(img_path)

    for rotation in [0, 90, 270]:
        rotated   = pil_img.rotate(rotation, expand=True) if rotation else pil_img
        processed = preprocess_for_ocr(rotated, min_height=min_height)

        for cfg in _CONFIGS:
            raw_text = pytesseract.image_to_string(processed, config=cfg)
            found = []
            for m in _FIG_LABEL.finditer(" ".join(raw_text.split())):
                num = re.sub(r'\s+', '', m.group(1)).upper()
                key = f"FIG. {num}"
                if (key, rotation) not in found:
                    found.append((key, rotation))

            if found:
                return found   # ← exits both the config loop and rotation loop

    return []

In [70]:
def match_figs_by_ocr(image_paths, best_fig_desc, verbose: bool = False) -> dict:
    """
    Returns dict mapping FIG key -> {'path': ..., 'rotation': ...}
    """
    if isinstance(image_paths, str):
        image_paths = ast.literal_eval(image_paths)
    if isinstance(best_fig_desc, str):
        best_fig_desc = ast.literal_eval(best_fig_desc)

    target_keys = set(best_fig_desc.keys())
    result = {}

    for path in image_paths:
        if not Path(path).exists():
            if verbose:
                print(f"  [SKIP] missing file: {path}")
            continue

        try:
            found_labels = extract_fig_labels(path)
        except Exception as e:
            if verbose:
                print(f"  [ERROR] {path}: {e}")
            continue

        for (label, rotation) in found_labels:
            if label in target_keys and label not in result:
                result[label] = {'path': path, 'rotation': rotation}
                if verbose:
                    print(f"  [MATCH] {label} → {Path(path).name} (rotation={rotation}°)")

        if result.keys() == target_keys:
            break

    return result

In [93]:
def random_row_index(df):
    return np.random.randint(0, len(df))

idx = random_row_index(Impact_df)

In [94]:
idx

2871

In [95]:
Impact_df['best_fig_desc'].iloc[idx]

{'FIG. 1': 'FIG. 1 is a front top perspective view of a first embodiment of my invention.',
 'FIG. 2': 'FIG. 2 is a rear bottom perspective view of FIG. 1 .',
 'FIG. 9': 'FIG. 9 is a front top perspective view of a second embodiment of my invention.',
 'FIG. 10': 'FIG. 10 is a rear bottom perspective view of FIG. 1 .'}

In [96]:
best_figs = match_figs_by_ocr(Impact_df['image_paths'].iloc[idx], Impact_df['best_fig_desc'].iloc[idx], verbose=True)
print(best_figs)

  [MATCH] FIG. 9 → USD0970065-20221115-D00006.TIF (rotation=270°)
  [MATCH] FIG. 10 → USD0970065-20221115-D00007.TIF (rotation=270°)
{'FIG. 9': {'path': 'impact_dataset/2022/USD0970065-20221115/USD0970065-20221115-D00006.TIF', 'rotation': 270}, 'FIG. 10': {'path': 'impact_dataset/2022/USD0970065-20221115/USD0970065-20221115-D00007.TIF', 'rotation': 270}}
